<a id="unit-u.e122a2c8b7694435ba5332ae9f648de4"></a>

<div>
  <div>09-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Шардтау, репликация және үлестірілген орындауды бақылау</div>
  <p>Үш Frontend және екі Backend бастап, үшінші BE қосып, бар репликалардың соған жылжуын бақылаңыз; тікелей FE қосылымының жүктеме теңгерушісі еместігін дәлелдеңіз, күй тексеретін HAProxy кірісін қосыңыз, бір агрегатты орындалуда таңдалған BE Scan тапсырмаларымен қадағалаңыз, жазудан кейін оралған репликаның қуып жетуін бақылаңыз және Bucket Shuffle мен Colocate Join салыстырыңыз.</p>
  <span>Doris 4.1.3 · FE сайлауы · HAProxy · Bucket · Tablet · Реплика теңгеруі · Query Profile · Реплика нұсқасы · Bucket Shuffle · Colocate</span>
</div>

**9-модуль: Apache Doris жүйесінде шардтау, репликация және үлестірілген орындау** · [Сабақ](course.md) · [Тест](quiz09_distribution_and_availability.ipynb)

Шамамен 70 минут, алғашқы дерек жүктеуі баяу желіде болса сәл көбірек жоспарлаңыз.

Ұяшықтарды ретімен орындаңыз. Бұл жұмыс тек оқшауланған `doris-rta-m09` Compose жобасын және `m09_*` нысандарын басқарады. Тіркелгі қажет етпейтін ашық курс S3 bucket-інен бір шектелген аралық оқиды және S3 ешқашан өзгертпейді. 1–3-бөлімдер BE кеңейтуін, FE бағыттауын, үлестірілген Scan дәлелін, BE қалпына келуін және Join тарату салыстыруын қамтиды.

<a id="unit-u.677643a620ba488990b67671d6fc03e2"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

Келесі ұяшықты 1-бөлімге дейін және әр kernel қайта іске қосылуынан кейін орындаңыз. Ол курс түбірін тауып, ортақ `DorisLab` көмекшісін жүктейді және Compose пен HAProxy endpoint мәндерін тіркейді. Контейнер бастамайды не тоқтатпайды.

Көмекші құрал тек бар 9-модуль жобасын тексеру үшін тікелей `fe-1` endpoint арқылы басталады. Кейінгі ұяшықтар ағымдағы Master, HAProxy және Profile түсіруін үйлестіретін FE арқылы әдейі қайта қосылады.


In [ ]:
from pathlib import Path
import shlex
import sys

import pandas as pd

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab
from doris_course.environment import (
    M09_BE_HTTP_PORT,
    M09_FE_HTTP_PORTS,
    M09_FE_QUERY_PORTS,
    M09_PROXY_PORT,
    M09_PROXY_STATS_PORT,
)
from doris_course.ui import card, show_frame

MODULE_DIR = COURSE_ROOT / "level3/module09-sharding-replication"
COMPOSE_FILE = MODULE_DIR / "compose.multinode.yml"
# Host ports: FE query 19130-19132, FE HTTP 18130-18132, HAProxy 19230 and 18230,
# unless DORIS_RTA_M09_* environment variables set others (see doris_course/environment.py).
PROXY_STATS_URL = f"http://127.0.0.1:{M09_PROXY_STATS_PORT}/stats;csv"
PROXY_CONTAINER = "doris-rta-m09-haproxy-1"
PROXY_PORT = M09_PROXY_PORT
EXPECTED_EVENTS = 12_062

FE_ENDPOINTS = {
    "172.31.91.40": {"service": "fe-1", "port": M09_FE_QUERY_PORTS["fe-1"], "http_port": M09_FE_HTTP_PORTS["fe-1"]},
    "172.31.91.41": {"service": "fe-2", "port": M09_FE_QUERY_PORTS["fe-2"], "http_port": M09_FE_HTTP_PORTS["fe-2"]},
    "172.31.91.42": {"service": "fe-3", "port": M09_FE_QUERY_PORTS["fe-3"], "http_port": M09_FE_HTTP_PORTS["fe-3"]},
}
FE1 = FE_ENDPOINTS["172.31.91.40"]
M09_HOST_PORTS = sorted({
    *M09_FE_QUERY_PORTS.values(), *M09_FE_HTTP_PORTS.values(),
    M09_BE_HTTP_PORT, M09_PROXY_PORT, M09_PROXY_STATS_PORT,
})
M09_PORT_FILTER = " ".join(f"-iTCP:{port}" for port in M09_HOST_PORTS)

lab = DorisLab(
    lab_dir=COURSE_ROOT,
    connection_host="127.0.0.1",
    connection_port=FE1["port"],
    profile_http_host="127.0.0.1",
    profile_http_port=FE1["http_port"],
);


<a id="unit-u.3b2e06a546f7493caca7595ea8be5998"></a>

<a id="1-expand-the-be-layer-then-recover-a-failed-fe-connection"></a>

## 1. BE қабатын кеңейтіп, кейін үзілген FE қосылымын қалпына келтіру

Бұл бөлім бір қайталанатын топология ауысуымен жүреді:

1. Тікелей `fe-1` қосылымымен үш FE және екі BE бастаңыз. HAProxy тоқтаған қалады.
2. Сол екі тірі BE ішінде модульдің екі репликалық оқиға кестелерін қайта құрыңыз.
3. `be-3` бастап, Doris бар репликаларды жаңа нысанаға жылжытатынын бақылаңыз.
4. Ағымдағы Master түйініне тікелей қосылып, сол FE тоқтатыңыз және бар TCP қосылымының үзілгенін бақылаңыз.
5. Істемеген FE қайта бастамай HAProxy қосып, тұрақты порты арқылы алмастыру қосылымдарын ашыңыз, соңында тоқтаған FE қалпына келтіріңіз.

Контейнерді тоқтату жазылатын қабатын жоймайды. Тек модульге тиесілі оқиға кестелерін қайта құру BE күшпен жойылмай, 2 → 3 BE орналастыру экспериментін қайталанатын етеді.


<a id="unit-u.3d164570a35046cf93c303196d9bf854"></a>

<a id="11-start-the-services-with-exactly-two-live-bes"></a>

### 1.1 Қызметтерді дәл екі тірі BE арқылы іске қосу

Дайындау ұяшығы 1-зертханалық жұмыстың көрнекі барысына сай. Кез келген 9-модуль контейнерін бастамас бұрын Docker кемінде 4 CPU, 8 GB жад және 20 GiB бос диск орнын тексеріп, жеткіліксіз болса түсіндірмемен тоқтайды. Оқшауланған порттарды тексереді, Compose файлын тексереді, бекітілген образдарды қайта пайдаланады не жүктейді, сақталған FE метадеректеріне көпшілік қажет болса үш FE бірге бастайды, екі бастапқы BE үдерісін қайта бастайды және `be-3` пен HAProxy айқын тоқтаған күйде ұстайды. Қайта іске қосу контейнерлер мен деректерін жоймай, бұрынғы орындаулар сақтаған үдеріс жадын босатады.

Ноутбук `fe-1` түйініне тікелей қосылады. Барлық тіркелген BE метадеректерін үнсіз тексеріп, тек екі тірі жолды көрсетеді және дәл `be-1` мен `be-2` талап етеді. Бұрын тіркелген `be-3` FE метадеректерінде `Alive=false` болып қалуы мүмкін; бұл үшінші жұмыс істейтін BE емес.


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

 echo "[1/6] Verify the Docker client, daemon, CPUs, and memory"
 docker info >/dev/null
 docker compose version
 cpus=$(docker info --format '{{{{.NCPU}}}}')
 memory=$(docker info --format '{{{{.MemTotal}}}}')
 echo "Docker allocation: $cpus CPUs, $((memory / 1000000000)) GB memory"
 if [ "$cpus" -lt 4 ] || [ "$memory" -lt 8000000000 ]; then
   echo 'Module 9 runs three FEs and three BEs on one Docker host and needs at least 4 CPUs and 8 GB of memory.'
   echo 'Raise the limits in Docker Desktop > Settings > Resources, or free them on the Docker host, then rerun this cell.'
   exit 5
 fi

 echo "[2/6] Inspect the isolated Module 9 ports"
 if command -v lsof >/dev/null 2>&1; then
   lsof -nP {M09_PORT_FILTER} -sTCP:LISTEN || true
 fi

 echo "[3/6] Validate the course Compose file"
 docker compose -f "$COMPOSE_FILE" config --quiet
 if [ "$(docker inspect -f '{{{{.State.Running}}}}' doris-rta 2>/dev/null || true)" = "true" ]; then
   echo 'The single-node doris-rta sandbox from Modules 1-8 is running. Stop it first with: docker stop doris-rta'
   exit 4
 fi

 echo "[4/6] Reuse or pull the pinned images"
 docker image inspect apache/doris:all-in-one-4.1.3 >/dev/null 2>&1 || docker pull apache/doris:all-in-one-4.1.3
 docker image inspect haproxy:3.0.30-alpine3.24 >/dev/null 2>&1 || docker pull haproxy:3.0.30-alpine3.24

 echo "[5/6] Check free disk space for container data"
 free_kb=$(docker run --rm --entrypoint df apache/doris:all-in-one-4.1.3 -Pk / | awk 'NR == 2 {{print $4}}')
 echo "Free disk space for containers: $((free_kb / 1048576)) GiB"
 if [ "$free_kb" -lt $((20 * 1048576)) ]; then
   echo 'Module 9 needs at least 20 GiB of free disk space for container data.'
   echo 'Remove unused images and stopped containers (docker system df shows the usage) or enlarge the Docker disk, then rerun this cell.'
   exit 6
 fi

 echo "[6/6] Start three FEs with freshly restarted baseline BEs; keep HAProxy stopped"
 # Reset BE process memory retained by earlier runs. `stop` preserves each
 # container writable layer and all Doris data; the following `up` starts only be-1 and be-2.
 docker compose -f "$COMPOSE_FILE" stop be-1 be-2 be-3 haproxy >/dev/null 2>&1 || true
 if docker container inspect doris-rta-m09-fe-1-1 doris-rta-m09-fe-2-1 doris-rta-m09-fe-3-1 >/dev/null 2>&1; then
   # Existing FE metadata requires a majority. Start all FE containers concurrently,
   # then let Compose wait for health and start the two baseline BEs.
   docker start doris-rta-m09-fe-1-1 doris-rta-m09-fe-2-1 doris-rta-m09-fe-3-1 >/dev/null
 fi
 docker compose -f "$COMPOSE_FILE" up -d --wait fe-1 fe-2 fe-3 be-1 be-2
 docker compose -f "$COMPOSE_FILE" ps
""", title="Prepare the two-BE Module 9 environment");

assert not lab.container_running(PROXY_CONTAINER), (
    "HAProxy must be stopped after Section 1.1; inspect the Module 9 Compose project before continuing."
)

lab.connect(
    container="doris-rta-m09-fe-1-1",
    host="127.0.0.1", port=FE1["port"], http_port=FE1["http_port"],
)
lab.execute("CREATE DATABASE IF NOT EXISTS doris_course")
lab.execute("USE doris_course")

registered_backends = lab.query("SHOW BACKENDS")
expected_registered = len(registered_backends.index)
assert expected_registered in (2, 3)
backend_state = lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=expected_registered, expected_alive=2,
    title="BE metadata with exactly two live nodes", show=False,
)
live_backends = backend_state.loc[
    backend_state["Alive"].astype(str).str.lower() == "true",
    ["BackendId", "Host", "Alive", "TabletNum", "Version"],
].reset_index(drop=True)
show_frame("Exactly two BEs are alive before expansion", live_backends)
assert set(live_backends["Host"].astype(str)) == {"172.31.91.50", "172.31.91.51"}
assert live_backends["Version"].nunique() == 1

lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    expected_roles=("FOLLOWER",), require_same_version=True,
    title="Three joined FEs and the current Master",
);

# An Alive heartbeat can precede the first disk report.
lab.wait_for_backend_storage(
    expected_backends=expected_registered, expected_alive=2,
);


<a id="unit-u.f7b0722623dc4518aaac0c66675c837c"></a>

**Күтілетін нәтиже**

- Үш FE тірі және дәл біреуі `IsMaster=true` көрсетеді.
- Соңғы тірі BE кестесінде дәл екі жол бар: осы бекітілген Compose желісіндегі `172.31.91.50` және `172.31.91.51`.
- Қайталауда `SHOW BACKENDS` ішінде тіркелген `be-3` `Alive=false` болып та болуы мүмкін. Бастапқы кестелер қайта құрылғанда ол тірі орналастыру нысанасы болып саналмайды.
- `be-1` және `be-2` жаңа үдерістер ретінде басталады, сондықтан бұрынғы орындаулардың үдеріс жады босайды. Ешбір BE тіркеуі, контейнер жазылатын қабаты, Doris дерегі немесе S3 нысаны жойылмайды.


<a id="unit-u.08aa340a1ed140c69b039987a75f8458"></a>

<a id="12-add-be-3-and-observe-automatic-replica-migration"></a>

### 1.2 `be-3` қосып, автоматты реплика көшуін бақылау

Оқиға кестесі екі Partition, әрқайсысына алты Bucket және әр Tablet-ке екі реплика қолданады. Тек екі BE тірі кезде әр логикалық Tablet-тің әр тірі BE ішінде бір көшірмесі болуы тиіс. Әр орындау осы бақыланатын физикалық орналасудан басталуы үшін жұмыс тек модульге тиесілі оқиға кестесін қайта құрады.

Толық `m09_events` кестесі бар болса, уақытша қайта аталып, жергілікті 12,062 жолдық дереккөз ретінде қолданылады. Алғашқы орындау сол шектелген S3 аралығын оқиды. Оқиға кестесі екі BE ішінде дұрыс болғаннан кейін жұмыс ешбір реплика жылжыту пәрменінсіз `be-3` бастайды.


<a id="unit-u.a3d501073aff4437b3e2bdb9e3a3236c"></a>

<a id="rebuild-and-load-the-controlled-two-be-baseline"></a>

#### Бақыланатын екі BE бастапқы жиынын қайта құру және жүктеу

Бір ұяшық толық дайындықты орындайды: бекітілген жергілікті бастапқы жиынды қайта пайдалануға болатынын үнсіз тексереді, тек екі BE тірі кезде екі репликалық `m09_events` кестесін жасап, жүктейді. Алғашқы орындау шектелген S3 аралығын оқиды; қайталау жергілікті жиынды көшіріп, S3 қайта оқымайды. Аралықтағы 12,062 жол барлық 40 курс файлына таралған, сондықтан оларды табу үшін Doris шамамен 59 MB оқиды: баяу желіде осы ұяшықтың алғашқы орындалуы 1-зертханалық жұмыс жүктеуінің жартысынан көп уақыт алуы мүмкін.

Соңғы тексерулер белгілі жол санын және екі реплика анықтамасын тексереді. Келесі ұяшық Tablet метадеректерін күтіп, кеңейту экспериментіне қажетті ықшам реплика орналасуын көрсетеді.

In [ ]:
# Reuse a complete local baseline on reruns without displaying an intermediate table.
def complete_local_baseline(table_name):
    if not lab.table_exists(table_name):
        return False
    rows = lab.query(f"SELECT COUNT(*) AS row_count FROM {table_name}")
    return int(rows.iloc[0]["row_count"]) == EXPECTED_EVENTS

current_ready = complete_local_baseline("m09_events")
previous_ready = complete_local_baseline("m09_events_previous")

if current_ready:
    lab.execute("DROP TABLE IF EXISTS m09_events_previous FORCE")
    lab.execute("ALTER TABLE m09_events RENAME m09_events_previous")
    source_table = "m09_events_previous"
elif previous_ready:
    lab.execute("DROP TABLE IF EXISTS m09_events FORCE")
    source_table = "m09_events_previous"
else:
    lab.execute("DROP TABLE IF EXISTS m09_events FORCE")
    lab.execute("DROP TABLE IF EXISTS m09_events_previous FORCE")
    source_table = None

# Remove a skew-comparison table that older copies of this Lab created.
lab.execute("DROP TABLE IF EXISTS m09_events_skewed FORCE")
for table in ("m09_availability_events", "m09_availability_probe", "m09_users"):
    lab.execute(f"DROP TABLE IF EXISTS {table} FORCE")

lab.execute("""
CREATE TABLE m09_events (
    event_time DATETIME,
    event_id BIGINT,
    user_id BIGINT,
    product_id BIGINT,
    event_type VARCHAR(16),
    region VARCHAR(16),
    revenue DECIMAL(12,2)
)
DUPLICATE KEY(event_time, event_id)
PARTITION BY RANGE(event_time) (
    PARTITION p_first_half VALUES LESS THAN ("2020-03-01 03:30:00"),
    PARTITION p_second_half VALUES LESS THAN ("2020-03-01 07:00:00")
)
DISTRIBUTED BY HASH(user_id) BUCKETS 6
PROPERTIES ("replication_num" = "2")
""")

if source_table is None:
    event_insert = """
    INSERT INTO m09_events
    SELECT event_time, event_id, user_id, product_id, event_type, region, revenue
    FROM S3(
        "uri" = "s3://doris-public-dataset/doris-course/events/v1/events-*.parquet",
        "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
        "s3.region" = "us-east-1",
        "s3.credentials_provider_type" = "ANONYMOUS",
        "s3.connection.timeout" = "10000",
        "s3.connection.request.timeout" = "30000",
        "format" = "parquet"
    )
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 07:00:00'
    """
else:
    event_insert = f"""
    INSERT INTO m09_events
    SELECT event_time, event_id, user_id, product_id, event_type, region, revenue
    FROM {source_table}
    """

loaded_events = lab.insert(
    event_insert,
    title=("Load the bounded S3 events" if source_table is None else "Rebuild events from the committed local baseline"),
    expected_rows=EXPECTED_EVENTS,
    show=False,
)
assert loaded_events == EXPECTED_EVENTS
assert lab.table_replication_num("m09_events") == 2

In [ ]:
lab.wait_for_tablet_rows("m09_events", expected_rows=EXPECTED_EVENTS)
lab.wait_for_replica_health("m09_events", expected_alive=2, expected_complete=2)

before_events = lab.replica_distribution_summary("m09_events", show=False)

# The two-BE baseline must begin with 12 + 12 + 0 physical replicas.
def replica_count(frame, host):
    match = frame.loc[frame["host"].astype(str) == host, "stored replicas"]
    return int(match.iloc[0]) if not match.empty else 0

counts = [replica_count(before_events, host) for host in ("172.31.91.50", "172.31.91.51", "172.31.91.52")]
assert counts == [12, 12, 0]

before_stage = pd.DataFrame([{
    "stage": "Before be-3 starts",
    "be-1": replica_count(before_events, "172.31.91.50"),
    "be-2": replica_count(before_events, "172.31.91.51"),
    "be-3": replica_count(before_events, "172.31.91.52"),
    "total replicas": int(before_events["stored replicas"].astype(int).sum()),
}])
show_frame("Event-table replica placement before expansion", before_stage)
lab.show_statements("SHOW REPLICA DISTRIBUTION FROM m09_events", "SHOW BACKENDS")
lab.execute("DROP TABLE IF EXISTS m09_events_previous FORCE");

<a id="unit-u.e3e053e4d4f346029498ef1b4e49f7b4"></a>

**Күтілетін нәтиже**

- Көрсетілген жол `m09_events` үшін физикалық реплика орналасуын сипаттайды.
- 12 логикалық Tablet және әр Tablet-ке екі реплика бар, сондықтан кестеде 24 физикалық реплика.
- Тек `be-1` және `be-2` тірі кезде әрқайсысы әр Tablet-тің бір репликасын сақтайды: `12 + 12 + 0 = 24`.
- Бұл — бекітілген бастапқы нүкте. Келесі ұяшық `be-3` бастап, жаңа BE бар репликаларды алғанша Doris метадеректерін тексереді.


<a id="unit-u.07ac7581ed3e4783abb277cc907290ce"></a>

<a id="start-be-3-and-wait-for-automatic-replica-migration"></a>

#### `be-3` бастап, автоматты реплика көшуін күту

`be-3` бастау тек тірі орналастыру нысанасын қосады; ешбір кесте қайта жүктелмейді және реплика жылжыту пәрмені берілмейді. Көмекші құрал үш BE де кемінде бір реплика сақтап, жиынтық 24 болып қалғанша секундына бір рет, ең көбі үш минут реплика метадеректерін тексереді. Содан кейін нәтиже сақталған «бұрын» жолының жанына қойылады.


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" up -d --wait be-3
""", title="Start the third BE")

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3, require_same_version=True,
    title="Three live BEs after expansion", show=False,
)
lab.wait_for_backend_storage(expected_backends=3, show=False)

expanded_events = lab.wait_for_replica_distribution(
    "m09_events",
    expected_backends=3, expected_total_replicas=24,
    timeout_seconds=180, show=False,
)

counts = [replica_count(expanded_events, host) for host in ("172.31.91.50", "172.31.91.51", "172.31.91.52")]
assert all(count > 0 for count in counts)
assert sum(counts) == 24

logical_rows = lab.query("SELECT COUNT(*) AS row_count FROM m09_events")
assert int(logical_rows.iloc[0]["row_count"]) == EXPECTED_EVENTS

after_stage = pd.DataFrame([{
    "stage": "After automatic migration",
    "be-1": replica_count(expanded_events, "172.31.91.50"),
    "be-2": replica_count(expanded_events, "172.31.91.51"),
    "be-3": replica_count(expanded_events, "172.31.91.52"),
    "total replicas": int(expanded_events["stored replicas"].astype(int).sum()),
}])
show_frame(
    "Existing event replicas move onto the new BE",
    pd.concat([before_stage, after_stage], ignore_index=True),
)
lab.show_statements("SHOW REPLICA DISTRIBUTION FROM m09_events", "SHOW BACKENDS");

<a id="unit-u.17b44db9fcf24f8494be47355ced7bf6"></a>

**Күтілетін нәтиже**

- Бірінші жол `12, 12, 0` болып қалады; екіншісінде үш BE үшін де оң реплика саны бар. Көшуден кейінгі нақты сандар өзгеруі мүмкін.
- Кестеде әлі 12 Tablet және әр Tablet-ке екі реплика болғандықтан, жиынтық 24 қалады. Тағы бір BE бастау `replication_num` өзгертпейді.
- `be-3` басталғаннан кейін `ADMIN REBALANCE`, қолмен Tablet көшіру немесе кестені қайта жүктеу жоқ. Өзгерген орналасу — Doris бар репликаларды фонда жылжытқанының тікелей дәлелі.
- Код реплика тасымалынан кейін `m09_events` әлі 12,062 логикалық жол қамтитынын тексереді.


<a id="unit-u.0a6760674bba4ac882c209859375fc03"></a>

<a id="13-break-a-direct-fe-connection-then-recover-through-haproxy"></a>

### 1.3 Тікелей FE қосылымын үзіп, HAProxy арқылы қалпына келтіру

Бірінші код ұяшығы тек ағымдағы FE күйін көрсетеді. Қате бермейді, қосылымды өзгертпейді, HAProxy бастамайды және үдеріс тоқтатпайды.

Екінші ұяшық өзгертетін қадамға иелік етеді. Бірдеңені тоқтатардың дәл алдында үш тірі FE талап етіп, HAProxy контейнері тоқтағанын растайды. Содан кейін ағымдағы Master FE түйініне тікелей қосылып, нақты тікелей портты көрсетеді, сол FE тоқтатады және бастапқы қосылым арқылы мағыналы `m09_events` санауын қайта орындайды. Күтілетін нәтиже — қосылым үзілуі, өйткені басқа FE сол TCP socket иелігін ала алмайды.

Сол үзіліс байқалғаннан кейін ғана жұмыс HAProxy `--no-deps` арқылы бастайды, сондықтан Compose істемеген FE қайта бастай алмайды. Тұрақты прокси порты арқылы жаңа қосылымдар содан кейін қалған екі FE-нің біріне жете алады.


In [ ]:
fe_state = lab.sql(
    "SHOW FRONTENDS", title="FE state before stopping the direct FE",
    columns=["Host", "Role", "IsMaster", "Join", "Alive", "Version"],
)


In [ ]:
pre_stop_state = lab.query("SHOW FRONTENDS")
pre_stop_alive = pre_stop_state["Alive"].astype(str).str.lower() == "true"
if int(pre_stop_alive.sum()) != 3:
    raise RuntimeError(
        "Stopping an FE requires three live FEs. Rerun Section 1.1 first. "
        "No FE was stopped."
    )
if lab.container_running(PROXY_CONTAINER):
    raise RuntimeError(
        "HAProxy must be stopped before the direct-connection failure test. "
        "Rerun Section 1.1 first. No FE was stopped."
    )

master_rows = pre_stop_state.loc[
    pre_stop_state["IsMaster"].astype(str).str.lower() == "true"
]
if len(master_rows.index) != 1:
    raise RuntimeError("Expected exactly one Master FE. No FE was stopped.")
master_host = str(master_rows.iloc[0]["Host"])
master_endpoint = FE_ENDPOINTS[master_host]
master_service = master_endpoint["service"]

lab.connect(
    container=f"doris-rta-m09-{master_service}-1",
    host="127.0.0.1",
    port=master_endpoint["port"],
    http_port=master_endpoint["http_port"],
)
card(
    f"127.0.0.1:{master_endpoint['port']} → {master_service}; HAProxy port {PROXY_PORT} is stopped.",
    "info", "Direct FE selected for the failure test",
)

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop {shlex.quote(master_service)}
""", title=f"Stop the directly connected FE: {master_service}")

lab.expect_current_connection_failure(
    "SELECT COUNT(*) AS event_rows FROM m09_events",
    title="The existing direct FE connection cannot migrate",
);


<a id="unit-u.2691f7dde51f4b6dae60019d1ac6069f"></a>

**Күтілетін нәтиже**

- Бірінші кесте — бақылау ғана. Үштен аз тірі FE көру ортаны өзгертпейді; келесі ұяшық басқа FE тоқтатудан бас тартады.
- Үш тірі FE және тоқтаған HAProxy кезінде екінші ұяшық өзінің айқын тікелей FE жолын (әдепкі `127.0.0.1:19130`–`19132` немесе бапталған балама) анықтап, содан кейін сол FE тоқтатады.
- Сол қосылымды `SELECT COUNT(*) FROM m09_events` үшін қайта пайдалану ықшам `connection failed` нәтижесін береді. Басқа FE Master болуы бар TCP socket иелігін ала алмайды.


<a id="unit-u.82d4ede883bd425686a4a3d3bbe5caee"></a>

<a id="start-haproxy-and-establish-replacement-connections"></a>

#### HAProxy бастап, алмастыру қосылымдарын орнату

HAProxy енді істемеген FE тоқтаған күйде басталады. Үш нысананы да тексеріп, әр **жаңа** қосылымды тек `UP` FE түйініне жібереді. Төрт үнсіз `CONNECTION_ID()` тексеруі төрт бөлек сеанс орнатып, round-robin санағыш өзгерісін көрінетін етеді; қосылым ID мәндері шығарылмайды.

HAProxy санағыштары қосылым таралуын көрсетеді. `SHOW FRONTENDS` бөлек түрде қалған екі FE арасында Doris дәл бір Master сақтағанын көрсетеді.


In [ ]:
failed_services = [
    endpoint["service"]
    for endpoint in FE_ENDPOINTS.values()
    if not lab.container_running(f"doris-rta-m09-{endpoint['service']}-1")
]
if len(failed_services) != 1:
    raise RuntimeError(
        "Expected exactly one stopped FE container before starting HAProxy. "
        "Run the preceding direct-connection failure cell first."
    )
failed_service = failed_services[0]

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" up -d --wait --no-deps haproxy
""", title="Start HAProxy while the directly connected FE remains stopped")

proxy_before = lab.wait_for_proxy_backend_status(
    PROXY_STATS_URL, failed_service, "DOWN",
    title=f"HAProxy excludes the stopped target: {failed_service}", show=False,
).set_index("FE target")
connection_ids = []
for _ in range(4):
    lab.connect(
        container=PROXY_CONTAINER,
        host="127.0.0.1", port=PROXY_PORT,
        http_host="127.0.0.1", http_port=FE1["http_port"],
    )
    connection = lab.query("SELECT CONNECTION_ID() AS connection_id")
    connection_ids.append(int(connection.iloc[0]["connection_id"]))
assert len(set(connection_ids)) == 4

proxy_after = lab.proxy_backend_summary(PROXY_STATS_URL, show=False).set_index("FE target")
connection_change = proxy_after[["proxy status"]].copy()
connection_change["new connections"] = (
    proxy_after["total connections"] - proxy_before["total connections"]
)
show_frame("New connections after one FE stopped", connection_change.reset_index())
assert int(connection_change.loc[failed_service, "new connections"]) == 0
assert int(connection_change["new connections"].sum()) >= 4

lab.execute("USE doris_course")
lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=2, expected_masters=1,
    title="A surviving Follower has become Master",
)
proxy_result = lab.sql(
    "SELECT COUNT(*) AS event_rows FROM m09_events",
    title="Course events remain queryable through HAProxy",
)
assert int(proxy_result.iloc[0]["event_rows"]) == EXPECTED_EVENTS;


<a id="unit-u.5745b9575c084197a9a9c87cfe17ac34"></a>

**Күтілетін нәтиже**

- HAProxy санағыш кестесі қалған екі FE нысанасына бөлінген кемінде төрт жаңа қосылым көрсетеді; тоқтаған нысана айырмасы нөл.
- `SHOW FRONTENDS` екі тірі FE және дәл бір Master көрсетеді. Бұл — Doris FE сайлау дәлелі; қосылым таралуы HAProxy ішінен келеді.
- Бастапқы тікелей қосылым үзілгеннен кейін HAProxy порты (әдепкі `19230`) арқылы `m09_events` саны 12,062 қалады, бұл жаңадан орнатылған қосылым тұрақты прокси endpoint арқылы қалған FE түйініне жете алатынын дәлелдейді.


<a id="unit-u.75f161cb139940cd9404ffe73427aa57"></a>

<a id="14-restore-the-stopped-fe"></a>

### 1.4 Тоқтаған FE қалпына келтіру

Бұрынғы Master үдерісін қайта іске қосып, Doris үш тірі FE көрсеткенше және HAProxy оралған нысананы `UP` белгілегенше күтіңіз. Оралған FE қайта Master болуы міндетті емес. Соңғы тікелей қосылым нақты ағымдағы Master пайдаланады, тек кейінгі Profile алу үшін белгілі HTTP үйлестіруші endpoint қажет.


In [ ]:
stopped_services = [
    endpoint["service"]
    for endpoint in FE_ENDPOINTS.values()
    if not lab.container_running(f"doris-rta-m09-{endpoint['service']}-1")
]
if len(stopped_services) > 1:
    raise RuntimeError(
        "More than one FE container is stopped. Restore the three-FE baseline from Section 1.1."
    )

if len(stopped_services) == 1:
    recovery_service = stopped_services[0]
    lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" start {shlex.quote(recovery_service)}
""", title=f"Restart the stopped FE: {recovery_service}")

    recovered_proxy = lab.wait_for_proxy_backend_status(
        PROXY_STATS_URL, recovery_service, "UP", show=False,
    )
    proxy_health = recovered_proxy[["FE target", "proxy status", "last check"]].copy()
    proxy_health["last check"] = proxy_health["last check"].replace({
        "L4OK": "L4OK — TCP connection to FE port 9030 succeeded",
    })
    show_frame(
        f"HAProxy sees the restored FE as healthy: {recovery_service}",
        proxy_health,
    )
else:
    card("All three FEs are already alive; no container restart was needed.", "skip", "FE already restored")

recovered_fes = lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    title="All three FEs after the stopped FE returns",
)

coordinator_host = str(
    recovered_fes.loc[recovered_fes["IsMaster"] == "true", "Host"].iloc[0]
)
coordinator = FE_ENDPOINTS[coordinator_host]
lab.connect(
    container=f"doris-rta-m09-{coordinator['service']}-1",
    host="127.0.0.1", port=coordinator["port"], http_port=coordinator["http_port"],
)
lab.execute("USE doris_course");


<a id="unit-u.2d0b32e60eaa4ed7aafe65b3a132e894"></a>

**Күтілетін нәтиже**

- `proxy status=UP` FE жаңа клиент қосылымын алуға жарамды дегенді білдіреді.
- `last check=L4OK` HAProxy сол FE MySQL query `9030` портына Layer 4 TCP қосылымын сәтті ашты дегенді білдіреді. SQL орындамайды және Master анықтамайды.
- Бұл қалпына келу кестесі қосылым санағыштарын әдейі түсіреді. Санағыш айырмалары бұрынғы бағыттау салыстыруына тиесілі; жинақталған жиынтықтар FE қалпына келуін дәлелдемейді.
- Үш FE де `Alive=true` күйіне оралады, ал дәл біреуі Master қалады. FE қайта бастау болашақ қолжетімділікті қалпына келтіреді; 1.3-бөлімде үзілген TCP сеансын тірілтпейді.


<a id="unit-u.c860d1032de44546afc36e75aece9fda"></a>

<a id="2-observe-distributed-scan-work-and-replica-recovery"></a>

## 2. Үлестірілген сканерлеу жұмысын және реплика қалпына келуін бақылау

1-бөлім әр оқиға Tablet-іне екі репликасы бар үш орналастыру нысанасын орнатты. 2-бөлім енді екі сұрақты ажыратады:

1. Бір үлестірілген оқиға сұрауын қай BE репликалары нақты орындайды?
2. Бір BE тоқтағанда үш репликалық оқиға кестесі көпшілік жазуын қабылдай ала ма және оралған реплика қуып жете ме?

Бірінші сұраққа `m09_events` жауап береді. Екіншісіне бекітілген 12,062 жолдық оқиға жиынын өзгертпей модульге тиесілі шағын `m09_availability_events` кестесі жауап береді.


<a id="unit-u.2fbb34a314f0455bb8b86a8639d8f529"></a>

<a id="21-inspect-the-event-tables-physical-layout"></a>

### 2.1 Оқиға кестесінің физикалық орналасуын тексеру

`storage_layout_summary` логикалық Partition, Bucket және Tablet құрылымын әрі сақтау қабатының ағымдағы жол ауқымын көрсетеді. Кейінгі сұрау қай репликаны оқитынын анықтамайды.


In [ ]:
lab.wait_for_tablet_rows("m09_events", expected_rows=12_062)
lab.storage_layout_summary(["m09_events"]);


<a id="unit-u.b7b8c6f48275422893740ea812b81761"></a>

**Күтілетін нәтиже**

- Жалғыз жол екі Partition, әрқайсысына алты Bucket, 12 логикалық Tablet, әр Tablet-ке екі реплика және 12,062 логикалық жол көрсетеді.
- `rows / tablet` — 12 логикалық Tablet арасындағы ауқым. Ол келесі сұрауда әр BE қанша жол сканерлейтінін емес, деректер орналасуын сипаттайды.
- Әр Tablet-ке екі реплика Doris оқи алатын екі физикалық көшірме бар дегенді білдіреді. Сұрау әр қажетті Tablet-тің бір көшірмесін таңдайды; екеуін де сканерлемейді.


<a id="unit-u.865c122d8bf84a34a841987e60a716cf"></a>

<a id="22-observe-one-aggregate-across-the-selected-be-scan-tasks"></a>

### 2.2 Бір агрегатты таңдалған BE Scan тапсырмалары бойынша бақылау

Сұрау `m09_events` ішінен сегіз аймақ жиынтығын есептейді. `EXPLAIN` жоспарланған фрагменттерді, Exchange және таңдалған Tablet ауқымын көрсетеді. Қысқартылған `DetailProfile` нақты орындалған әр Scan тапсырмасы үшін BE хостын, Tablet ID және `ScanRows` сақтайды. Осы шағын оқу сұрауы үшін жұмыс әр BE ішіндегі әр Fragment Pipeline тапсырма параллельдігін `1` орнатады. Бұл сол Pipeline қайталанатын тапсырма даналарын азайтып, нәтижеде қатысатын BE хосттарын айқындайды. Сұрауды бір BE-мен шектемейді және бүкіл сұрауды бір ағынды етпейді.

Көмекші уақытша Profile level 2 қосып, SQL Cache, Query Cache және Condition Cache өшіреді. Кейін сеанс баптауларын қалпына келтіреді. Сақтау және операциялық жүйе кэштері тазартылмайды, сондықтан бұл суық кэш уақыт сынағы емес.


In [ ]:
regional_query = """
SELECT region, COUNT(*) AS event_count, SUM(revenue) AS total_revenue
FROM m09_events
GROUP BY region
ORDER BY region
"""
regional_evidence = lab.capture_query(
    regional_query, settings={"parallel_pipeline_task_num": "1"}
)
regional_evidence.show_result("Regional totals from m09_events")
regional_evidence.show_distributed_plan("Planned fragments and scan scope")
regional_evidence.show_distributed_scan_profile("Actual BE scan tasks selected for this query")

assert int(regional_evidence.rows["event_count"].sum()) == 12_062


<a id="unit-u.46a30ff8dd1c4cdfabddbf8991485e3f"></a>

**Күтілетін нәтиже**

Алдымен бизнес жауабын растаңыз. Нәтижеде сегіз аймақ жолы бар және олардың `event_count` мәндері 12,062 болып қосылады.

Қысқартылған `EXPLAIN` нәтижесін жоспарланған жұмыс ретінде оқыңыз. Құрылымы мынаған ұқсайды:

```text
PLAN FRAGMENT 0
  PARTITION: UNPARTITIONED
  VRESULT SINK
  VMERGING-EXCHANGE

PLAN FRAGMENT 1
  PARTITION: HASH_PARTITIONED: region[...]
  VAGGREGATE (merge finalize)
  VEXCHANGE

PLAN FRAGMENT 2
  PARTITION: RANDOM
  STREAM DATA SINK
    HASH_PARTITIONED: region[...]
  VAGGREGATE (update serialize)
  VOlapScanNode
    TABLE: doris_course.m09_events(m09_events)
    partitions=<selected>/<total>
    tablets=<selected>/<total>
    numNodes=<planned BE count>
```

`PLAN FRAGMENT 2` таңдалған Tablet сканерлеп, ішінара агрегаттау орындайды. `VEXCHANGE` ішінара аймақ топтарын тасымалдайды, ал жоғарғы агрегат жауапты аяқтайды. `tablets=12/12` әр Tablet-тің екі физикалық репликасын емес, 12 логикалық Tablet-ті білдіреді.

Содан кейін қысқартылған DetailProfile жазбасын нақты орындау дәлелі ретінде оқыңыз:

```text
Fragment 2:
  Pipeline 0(host=TNetworkAddress(hostname:<BE host>, port:9050)):
    PipelineTask(index=<task index>):
      OLAP_SCAN_OPERATOR(... table_name=m09_events(m09_events) ...):
        - TabletIds: [<selected Tablet replica IDs>]
        - ScanRows: <rows scanned by this task>
```

Осы өрістерге назар аударыңыз:

- Әр бөлек `host=TNetworkAddress(...)` — осы сұрауға Scan нақты орындаған BE. Әртүрлі BE мекенжайлары сұраудың бірнеше BE арасында үлестірілгенін көрсетеді; олардың тағайындалған фрагмент даналары қатар орындала алады.
- `TabletIds` тапсырмаға тағайындалған логикалық Tablet жұмысын анықтайды. Doris әр қажетті Tablet-ке бір қолжетімді реплика таңдайды; екі репликаны да оқымайды.
- `ScanRows` — сол тапсырма сканерлеген нақты жол саны.
- `SHOW REPLICA DISTRIBUTION` реплика сақтайтынын көрсетсе де, Profile ішінде жоқ BE осы орындауда `m09_events` сканерлемеген. Сақталған орындауда `.50` және `.51` қатысты, ал `.52` қатыспады.

Нақты қатысатын BE жиыны, Tablet ID, тапсырма топтауы және жол сандары басқа орналастыру не жоспарлау шешімінен кейін өзгеруі мүмкін. Сондықтан жұмыс үш BE де көрінуін талап етпейді және масштабтауды мәлімдеу үшін өткен уақыт қолданбайды. `EXPLAIN` мен бұл Profile қай FE сұрауды жоспарлағанын да анықтамайды; FE бағыттауы үшін 1-бөлім қосылым және HAProxy дәлелін пайдаланады.


<a id="unit-u.9580cbb3e4dd45ebba0affafdced6bc4"></a>

<a id="23-write-an-event-with-one-be-stopped"></a>

### 2.3 Бір BE тоқтағанда оқиға жазу

`m09_events` екі репликаға ие. Doris әдепкі көпшілікпен жазу ережесінде екі репликалық кестеге екі көшірме де қажет, сондықтан бір BE қолжетімсізде сәтті жазуды дәлелдеуге сай емес.

Сондықтан `m09_availability_events` бір логикалық Tablet және үш репликаға ие. Бірінші ұяшық әр BE тірі кезде бір бастапқы оқиға енгізіп, әр BE-де өзекті көшірме барын тексереді. Келесі ұяшық `be-2` тоқтатады; содан кейін `be-1` және `be-3` көпшілік құрып, бекітілген 12,062 жолдық оқиға жиынын өзгертпей бір белгілі сатып алу оқиғасын бекітеді.


In [ ]:
lab.execute("DROP TABLE IF EXISTS m09_availability_events FORCE")
lab.execute("""
CREATE TABLE m09_availability_events (
    event_id BIGINT,
    event_time DATETIME,
    user_id BIGINT,
    product_id BIGINT,
    event_type VARCHAR(16),
    region VARCHAR(16),
    revenue DECIMAL(12,2)
)
UNIQUE KEY(event_id)
DISTRIBUTED BY HASH(event_id) BUCKETS 1
PROPERTIES (
    "replication_num" = "3",
    "enable_unique_key_merge_on_write" = "true"
)
""")
lab.insert(
    """
    INSERT INTO m09_availability_events
    VALUES (9000001, '2020-03-01 07:00:00', 700001, 1001001, 'view', 'region_01', 0.00)
    """,
    title="Create the initial event while all three BEs are alive",
    expected_rows=1,
)
lab.wait_for_replica_health(
    "m09_availability_events", expected_alive=3, expected_complete=3,
)
availability_before = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="Before the stop: every BE has the current event-table version",
)
assert len(availability_before.index) == 3
assert (availability_before["current-version replicas"] == 1).all();


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop be-2
""", title="Stop be-2")

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=2,
    title="FE metadata after be-2 stops",
)
lab.execute("""
INSERT INTO m09_availability_events
VALUES (9000002, '2020-03-01 07:01:00', 700002, 1001002, 'purchase', 'region_01', 19.95)
""")
lab.sql("""
SELECT event_id, event_time, user_id, product_id, event_type, region, revenue
FROM m09_availability_events
WHERE event_id = 9000002
""", title="The purchase committed while be-2 is stopped")

availability_during = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="During the outage: be-2 retains an older event-table version",
)
assert int(availability_during["alive"].sum()) == 2
assert sorted(availability_during["current-version replicas"].tolist()) == [0, 1, 1];


<a id="unit-u.d3275bf744e44a61ba5ce3fbf382c5f0"></a>

**Күтілетін нәтиже**

Бизнес нәтижеде жаңадан бекітілген оқиға бар:

```text
event_id   event_time            user_id   product_id   event_type   region      revenue
9000002    2020-03-01 07:01:00   700002    1001002      purchase     region_01   19.95
```

Бұл `be-2` тоқтағаннан кейін жазылған сатып алуға сұрау жасауға болатынын дәлелдейді. Реплика кестесі содан кейін қай физикалық көшірменің сол жазуды жіберіп алғанын анықтайды.

Бекітілген нұсқа нөмірлерін күтпей, жолдарды салыстырып оқыңыз:

- `stored replicas` — сол BE ішінде тізілген физикалық көшірмелер саны. Үдерісті тоқтату репликаны жоймайтындықтан, `be-2` үшін `1` болып қалады.
- `replica versions` сол көшірмелер көрсеткен Doris `Version` қорытындысы. Бұл оқиға кестесінде бір Tablet және әр BE-ге бір көшірме бар, сондықтан әр жол ауқым емес, бір нұсқа нөмірін қамтиды.
- `current-version replicas` өз Tablet-і үшін қазір көрсетілген ең үлкен нұсқаға жеткен әрі сәтті қалыпты күйдегі көшірмелерді санайды.
- Екі тірі BE сол жаңарақ нұсқада болуы тиіс, мұнда `V_new` деп жазылған, әрқайсысы бір ағымдағы нұсқалы реплика көрсетеді.
- Тоқтаған `be-2` әлі бұрынғы `V_old` көрсетуі тиіс, мұнда `V_old < V_new`, және нөл ағымдағы нұсқалы реплика көрсетеді. `2` және `3` сияқты мәндер — орындауға тән мысалдар; тек осы рет маңызды.

Тең нұсқалар әртүрлі кезеңге тиесілі. Тоқтатуға дейін үш көшірме тең. `be-2` қайта басталғаннан кейін бірден оның нұсқасы әлі төмен не қалпына келу ноутбук үлгісіне дейін бітіп, тең болуы мүмкін. 2.4-бөлімнің шектелген күтуінен кейін үш нұсқа тең болуы тиіс. Сәтті жазудан кейін `be-2` тоқтаған күйде қалса, теңдік күтілмейді, өйткені сол көшірме жаңа нұсқаны ала алмайды.


<a id="unit-u.b22d4e6234f044a8ad9b67c57d9aedaf"></a>

<a id="24-restart-the-be-and-wait-for-version-recovery"></a>

### 2.4 BE қайта бастап, нұсқа қалпына келуін күту

`Alive=true` — тек алғашқы қалпына келу кезеңі. Мүмкін болса, ұяшық heartbeat кейін бірден реплика қорытындысын түсіреді, содан кейін үш көшірме де қалыпты және мақсатты нұсқада болғанша күтеді.

Қалпына келу ноутбук үлгі алып үлгергеннен тез болуы мүмкін. Дереу кесте барлық көшірме өзекті екенін көрсетсе, бұл жылдам қалпына келудің жарамды дәлелі; оны байқалған артта қалу кезеңі ретінде қайта жазуға болмайды.


In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" start be-2
""", title="Restart be-2")

lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3,
    title="All three BEs are alive again",
)
availability_on_return = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="Event-table versions immediately after be-2 returns",
)

lab.wait_for_replica_health(
    "m09_availability_events", expected_alive=3, expected_complete=3,
    timeout_seconds=180,
)
availability_recovered = lab.tablet_replica_backend_summary(
    "m09_availability_events",
    title="Recovery complete: every BE has the current event-table version",
)
assert availability_recovered["alive"].all()
assert (availability_recovered["current-version replicas"] == 1).all()
lab.sql("""
SELECT event_id, event_time, user_id, product_id, event_type, region, revenue
FROM m09_availability_events
WHERE event_id = 9000002
""", title="The recovered replicas retain the committed purchase");


<a id="unit-u.9e746ffa32874d7db6c43a210a365154"></a>

**Күтілетін нәтиже**

- `SHOW BACKENDS` алдымен үш BE үдерісі де тірі екенін растайды. Жалғыз heartbeat дәлелі `be-2` репликасын жөндегенін дәлелдемейді.
- Дереу реплика кестесінде `be-2` әлі төмен `replica versions` және нөл ағымдағы нұсқалы реплика көрсетуі мүмкін немесе үлгіге дейін қалпына келу бітіп, үнемі тірі екі BE-ге тең болуы мүмкін.
- `wait_for_replica_health` біткеннен кейін үш жол да сол реплика нұсқасын және бір ағымдағы нұсқалы репликаны көрсетуі тиіс. Бизнес сұрау `9000002` оқиғасын, `region_01` ішіндегі `19.95` түсімді `purchase` әлі қайтаруы тиіс.
- Бұл — осы бақыланатын оқиға кестесінің нұсқа қалпына келуінің дәлелі. Хостаралық апаттан қалпына келуді дәлелдемейді, өйткені барлық контейнер бір Docker хосты мен файлдық жүйесін әлі ортақ пайдаланады.


<a id="unit-u.db16be9841784ef5ad3cd72fd38010e5"></a>

<a id="3-reduce-join-movement-with-bucket-shuffle-and-colocate"></a>

## 3. Bucket Shuffle және Colocate арқылы Join тасымалын азайту

Оқиға қызметі пайдаланушы сегменті бойынша белсенділікті жиі есептейді. Оқиға кестесі `user_id` бойынша Hash таратылған, бірақ шағын пайдаланушы өлшемі әдепкіде әлі Broadcast болуы мүмкін. Бұл бөлім желі талабын көрсету үшін Shuffle салыстыруын әдейі сұрап, кейін тұрақты кесте орналасуын өзгертеді және Doris Colocate таңдауға мүмкіндік береді.

Екі эксперимент те бірінші оқиға Partition бойынша сол бизнес сұраққа жауап береді:

1. оқиға жолдарын әр `user_id` бойынша қорытындылау;
2. әр пайдаланушыны `new`, `returning` немесе `loyal` сегментіне біріктіру;
3. сегмент бойынша оқиға саны мен түсімді қайтару.

Әр пайдаланушы агрегаттауы жоспар дәлелінің бөлігі. Ол `user_id` бойынша Hash таратуды сақтап, Bucket Shuffle үшін қайта пайдаланылатын бір орналасу береді. Бұл — Doris Bucket Shuffle құжатында сипатталғандай аралық қатынас түрі.

<a id="unit-u.a1838a992a324b4a8101448a0a1d2f94"></a>

<a id="31-reuse-the-event-side-layout-with-bucket-shuffle"></a>

### 3.1 Bucket Shuffle арқылы оқиға жағы орналасуын қайта пайдалану

Бірінші `m09_users` анықтамасы Join кілті бойынша әдейі **таратылмаған**. Ол RANDOM таратуды қолданады, ал оқиға жағы аралық нәтижесі `user_id` бойынша Hash таратылған. `[shuffle]` hint бұл шағын өлшемді Broadcast салыстыруына айналдырмайды; Bucket Shuffle кепілдемейді, сондықтан нақты Doris 4.1.3 жоспары `BUCKET_SHUFFLE` көрсетпесе, көмекші құрал ұяшықты қабылдамайды.

`m09_events` кестесін алдыңғы Colocation Group ішінен шығару орнатуды қайта орындауға мүмкіндік береді. Тек модульге тиесілі пайдаланушы кестесі қайта құрылады.

In [ ]:
lab.execute("DROP TABLE IF EXISTS m09_users FORCE")
lab.execute('ALTER TABLE m09_events SET ("colocate_with" = "")')
lab.execute("""
CREATE TABLE m09_users (
    user_id BIGINT NOT NULL,
    user_segment VARCHAR(16) NOT NULL
)
DUPLICATE KEY(user_id, user_segment)
DISTRIBUTED BY RANDOM BUCKETS 6
PROPERTIES ("replication_num" = "2")
""")

lab.insert("""
INSERT INTO m09_users
SELECT
    user_id,
    CASE MOD(user_id, 3)
        WHEN 0 THEN 'new'
        WHEN 1 THEN 'returning'
        ELSE 'loyal'
    END AS user_segment
FROM (SELECT DISTINCT user_id FROM m09_events) users
""", title="Build the user segment dimension", expected_rows=6706);

<a id="unit-u.4b723017e7ec49cf81ed841eea126d81"></a>

Сұрау `m09_events` ішіндегі бір Partition бөлімін сақтайды: `p_first_half`. Ішкі агрегаттау оның оқиға жолдарын әр пайдаланушыға бір аралық жолға айналдырады. Соңғы агрегаттау содан кейін сұралған сегмент жиынтықтарын қайта береді.

In [ ]:
bucket_shuffle_query = """
SELECT
    u.user_segment,
    SUM(e.event_count) AS event_count,
    SUM(e.total_revenue) AS total_revenue
FROM (
    SELECT
        user_id,
        COUNT(*) AS event_count,
        SUM(revenue) AS total_revenue
    FROM m09_events
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 03:30:00'
    GROUP BY user_id
) e
INNER JOIN [shuffle] m09_users u
    ON e.user_id = u.user_id
GROUP BY u.user_segment
ORDER BY u.user_segment
"""

bucket_shuffle_result = lab.sql(
    bucket_shuffle_query,
    title="Segment totals with the non-colocated user layout",
)
lab.explain_join_distribution(
    "EXPLAIN " + bucket_shuffle_query,
    expected_strategy="bucket_shuffle",
    title="Bucket Shuffle: Join and Exchange evidence",
);

<a id="unit-u.666af99f433b4190abed8e5793b8ff5b"></a>

**Күтілетін нәтиже**

Бекітілген үлгінің әр орындалуында бизнес нәтиже сол үш сегмент жолын қамтиды:

```text
user_segment   event_count   total_revenue
loyal          1098          100089.35
new            1177          113473.45
returning       994          102868.16
```

Қысқартылған жоспарды Join амалынан сыртқа қарай оқыңыз:

```text
VHASH JOIN
|  join op: INNER JOIN(BUCKET_SHUFFLE)
|  equal join conjunct: (user_id = user_id)
|----VEXCHANGE
...
TABLE: doris_course.m09_events(m09_events)
partitions=1/2 (p_first_half)
TABLE: doris_course.m09_users(m09_users)
```

- `partitions=1/2 (p_first_half)` оқиға предикаты осы тікелей сақтау орналасуын қайта пайдалануға қажетті бір физикалық Partition таңдағанын дәлелдейді.
- `BUCKET_SHUFFLE` Join оқиға жағының әр пайдаланушы Hash орналасуын сақтап, пайдаланушы жолдарын Join жанындағы `VEXCHANGE` арқылы сәйкес Bucket ұяшықтарына бағыттайтынын дәлелдейді.
- Соңғы `GROUP BY`, реттеу немесе нәтиже біріктіру үшін Join үстінде басқа Exchange әлі болуы мүмкін. Дәлел осы Join тікелей қажет ететін дерек тасымалына қатысты.
- `[shuffle]` — бақыланатын оқу салыстыруы. Онсыз оңтайландырғыш шағын пайдаланушы кестесін орынды Broadcast етуі мүмкін.

<a id="unit-u.767395ba3ea74ebd8dc591b27900a3dd"></a>

<a id="32-pre-position-both-sides-for-colocate-join"></a>

### 3.2 Colocate Join үшін екі жақты да алдын ала орналастыру

Қайталанатын оқиға–пайдаланушы Join үйлесімді орналасуды кесте жобасының шарты ете алады. Келесі ұяшық сол логикалық `m09_users` деректерін `HASH(user_id)`, алты Bucket, екі реплика және `m09_events` сияқты сол Colocation Group арқылы қайта құрады. `user_id` Unique Key болады; әр пайдаланушыда бір жол бар, сондықтан Join кірісі бірдей қалады.

Топ құрылып жатқанда Doris репликаларды жылжытуы мүмкін. Сондықтан жоспар қабылданбас бұрын `IsStable=true` тексеріледі. Бірдей DDL Colocate қолжетімділігінің жеткілікті дәлелі емес.

In [ ]:
lab.execute("DROP TABLE IF EXISTS m09_users FORCE")
lab.execute('ALTER TABLE m09_events SET ("colocate_with" = "m09_user_join_group")')
lab.execute("""
CREATE TABLE m09_users (
    user_id BIGINT NOT NULL,
    user_segment VARCHAR(16) NOT NULL
)
UNIQUE KEY(user_id)
DISTRIBUTED BY HASH(user_id) BUCKETS 6
PROPERTIES (
    "replication_num" = "2",
    "enable_unique_key_merge_on_write" = "true",
    "colocate_with" = "m09_user_join_group"
)
""")

lab.insert("""
INSERT INTO m09_users
SELECT
    user_id,
    CASE MOD(user_id, 3)
        WHEN 0 THEN 'new'
        WHEN 1 THEN 'returning'
        ELSE 'loyal'
    END AS user_segment
FROM (SELECT DISTINCT user_id FROM m09_events) users
""", title="Rebuild the same users in the Colocation Group", expected_rows=6706)

lab.wait_for_colocation_group(
    "m09_user_join_group",
    title="The event-user Colocation Group is stable",
);

In [ ]:
colocate_query = """
SELECT
    u.user_segment,
    SUM(e.event_count) AS event_count,
    SUM(e.total_revenue) AS total_revenue
FROM (
    SELECT
        user_id,
        COUNT(*) AS event_count,
        SUM(revenue) AS total_revenue
    FROM m09_events
    WHERE event_time >= '2020-03-01 00:00:00'
      AND event_time <  '2020-03-01 03:30:00'
    GROUP BY user_id
) e
INNER JOIN m09_users u
    ON e.user_id = u.user_id
GROUP BY u.user_segment
ORDER BY u.user_segment
"""

colocate_result = lab.sql(
    colocate_query,
    title="The same segment totals from the colocated layout",
)
lab.assert_same_rows(bucket_shuffle_result, colocate_result)
lab.explain_join_distribution(
    "EXPLAIN " + colocate_query,
    expected_strategy="colocate",
    title="Colocate: local Join evidence",
);

<a id="unit-u.48fee41c76f54f73a30feedfd455875a"></a>

**Күтілетін нәтиже**

Бір жолдық топ күйі алты Bucket, екі реплика және `stable=true` көрсетуі тиіс. Сегмент жиынтықтары 3.1-бөлімге дәл сәйкес болуы тиіс; деректер орналасуын өзгерту SQL жауабын өзгертпеуі тиіс.

Қысқартылған жоспарда енді мыналар болуы тиіс:

```text
VHASH JOIN
|  join op: INNER JOIN(COLOCATE)
|  equal join conjunct: (user_id = user_id)
...
TABLE: doris_course.m09_users(m09_users)
TABLE: doris_course.m09_events(m09_events)
partitions=1/2 (p_first_half)
```

Join фрагментінде Hash Join мен екі кіріс сканерлеуі арасында `VEXCHANGE` жоқ. Екі кестенің сәйкес `user_id` Bucket ұяшықтары бірге орналасқан, сондықтан бұл Join ешбір кірісті қайта таратпайды. Exchange соңғы сегмент агрегаттауы немесе нәтиже біріктіру үшін басқа жерде әлі болуы мүмкін.

Бұл дәлел тұрақты қайталанатын оқиға–пайдаланушы байланысының орналасу шешімін қолдайды. Colocate әрдайым жылдамырақ екенін дәлелдемейді: Bucket саны параллельдікті басқарады, Colocation Group қолдауы орналасуды шектейді және осы бір хостты Docker кластері өндірістік желі шығынын өлшей алмайды.

<a id="unit-u.038eb21471f148b198c5c7741e224cb7"></a>

<a id="4-practice-count-tablets-and-replicas-before-you-look"></a>

## 4. Жаттығу: Tablet және репликаларды қарамас бұрын санау

1.2-бөлім алты Bucket бар екі Partition және екі репликалы `m09_events` кестесінде 12 логикалық Tablet және 24 реплика барын көрсетті. Метадеректерін қарамас бұрын екі басқа кесте үшін сол есептеуді жасаңыз:

- 2.3-бөлімнің `m09_availability_events` кестесі: Partition тармағы жоқ, `DISTRIBUTED BY HASH(event_id) BUCKETS 1`, `replication_num = 3`;
- 3.2-бөлімнің `m09_users` кестесі: Partition тармағы жоқ, алты Bucket бар `HASH(user_id)`, `replication_num = 2`.

Әр кесте үшін логикалық Tablet санын, реплика санын және бір Tablet-тің екі репликасы сол BE ішінде бола алатынын болжаңыз. Содан кейін `SHOW TABLETS` арқылы тексеріп, әр BE репликаларын санаңыз.

In [ ]:
# Your code here

<a id="unit-u.c6b088d109f14c26a92012530aa23938"></a>

Келесі ұяшық — үлгі шешім; ол жиналған. Салыстыру үшін оны ашыңыз немесе орындаңыз.

In [ ]:
from doris_course.ui import show_frame

for table, tablets, copies in (("m09_availability_events", 1, 3), ("m09_users", 6, 2)):
    replicas = lab.sql(f"SHOW TABLETS FROM {table}", title=f"Replicas of {table}",
                       columns=["TabletId", "ReplicaId", "BackendId", "Version"])
    assert replicas["TabletId"].nunique() == tablets
    assert len(replicas) == tablets * copies
    # The copies of one Tablet are on different BEs.
    assert (replicas.groupby("TabletId")["BackendId"].nunique() == copies).all()
    show_frame(f"Replicas of {table} on each BE",
               replicas.groupby("BackendId").size().rename("replicas").reset_index());

<a id="unit-u.d16c53868fe04540a3f7e2c1ed5add05"></a>

**Күтілетін нәтиже**

| Кесте | Логикалық Tablet | Репликалар | Әр BE ішіндегі репликалар |
|---|---:|---:|---|
| `m09_availability_events` | 1 | 3 | Үш BE әрқайсысында 1 |
| `m09_users` | 6 | 12 | Сыналған орындауда үш BE әрқайсысында 4 |

Айқын бөлінбеген кестеде бір Partition бар, сондықтан логикалық Tablet саны Bucket санына тең: 1 × 1 және 1 × 6. Репликация көбейтеді: 3 және 12 реплика. Бір Tablet көшірмелері әрдайым әртүрлі BE ішінде, сондықтан үш репликаға үш BE қажет. Шешім тексерулері үнсіз өтеді. `TabletId`, `ReplicaId` және `BackendId` әр орындауда өзгеше.

<a id="unit-u.d0eea676479d46e5a26deaa095ccb6fa"></a>

<a id="stop-the-module-9-cluster"></a>

### 9-модуль кластерін тоқтату

Үш FE, үш BE және HAProxy үдерістерін босатқыңыз келгенде осы қосымша ұяшықты орындаңыз. Ол бар 9-модуль контейнерлерін, жазылатын қабаттарын және Doris деректерін жоймай тоқтатады. Кластерді пайдалануды жалғастырғыңыз келсе, өткізіп жіберіңіз.

In [ ]:
lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}
docker compose -f "$COMPOSE_FILE" stop
docker compose -f "$COMPOSE_FILE" ps -a
""", title="Stop the Module 9 cluster");

<a id="unit-u.5c20178fd4c74e479fb4b72a472717f0"></a>

**Күтілетін нәтиже:** 9-модуль қызметтері `Exited` көрсетеді. Жазылатын қабаттары мен Doris метадеректері қолжетімді қалады.

<a id="restart-the-module-9-cluster"></a>

### 9-модуль кластерін қайта іске қосу

Сол көп түйінді ортамен жалғастырғыңыз келгенде осы ұяшықты орындаңыз. Сақталған FE кластерінде кез келген FE қалпына келуді аяқтамас бұрын көпшілік қажет, сондықтан ұяшық бар үш FE контейнерін бірге бастайды, кейін үш BE мен HAProxy бастайды не қайта пайдаланады. Тұрақты HAProxy порты арқылы қайта қосылып, сақталған оқиға, пайдаланушы және үзілістегі жазу деректерін тексереді.

Қайта іске қосу ұяшығы идемпотентті: 9-модуль қызметтері тоқтаған, басталып жатқан немесе жұмыс істеп тұрған кезде орындала алады. macOS жүйесінде ортақ көмекші арқылы қажет болса Docker Desktop ашады; Linux жүйесінде ұяшықтан бұрын Docker Engine бастаңыз.

In [ ]:
lab.ensure_docker_ready()

lab.shell(f"""
set -euo pipefail
COMPOSE_FILE={shlex.quote(str(COMPOSE_FILE))}

if docker container inspect \
  doris-rta-m09-fe-1-1 \
  doris-rta-m09-fe-2-1 \
  doris-rta-m09-fe-3-1 >/dev/null 2>&1; then
  docker start \
    doris-rta-m09-fe-1-1 \
    doris-rta-m09-fe-2-1 \
    doris-rta-m09-fe-3-1 >/dev/null
fi

docker compose -f "$COMPOSE_FILE" up -d --wait \
  fe-1 fe-2 fe-3 be-1 be-2 be-3 haproxy
""", title="Restart the Module 9 cluster")

lab.connect(
    container=PROXY_CONTAINER,
    host="127.0.0.1", port=PROXY_PORT,
    http_host="127.0.0.1", http_port=FE1["http_port"],
)
lab.execute("USE doris_course")
lab.wait_for_node_state(
    "SHOW FRONTENDS", expected_total=3, expected_alive=3, expected_masters=1,
    title="Three FEs after restart", show=False,
)
lab.wait_for_node_state(
    "SHOW BACKENDS", expected_total=3, expected_alive=3,
    title="Three BEs after restart", show=False,
)
lab.sql("""
SELECT
    (SELECT COUNT(*) FROM m09_events) AS event_rows,
    (SELECT COUNT(*) FROM m09_users) AS user_rows,
    (SELECT COUNT(*) FROM m09_availability_events WHERE event_id = 9000002)
        AS outage_purchase_rows
""", title="Persisted Module 9 data after restart");

<a id="unit-u.a7fd501144c04a04a56f343789f9bc2e"></a>

**Күтілетін нәтиже:** үш FE және BE де қызметке оралады. Сақталған нәтиже **12,062** оқиға жолын, **6,706** пайдаланушы жолын және BE үзілісінде жазылған **1** сатып алуды көрсетеді.

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

FE сайлауын клиент бағыттауынан ажыраттыңыз, HAProxy арқылы тоқтаған FE айналасынан жаңа қосылымдар орнаттыңыз, екі BE реплика орналасуын үшінші BE түйініне кеңейттіңіз, бір үлестірілген сұрауға таңдалған BE Scan тапсырмаларын қадағаладыңыз, қалған реплика көпшілігі арқылы оқиға бекітіп, оралған репликаның қуып жетуін байқадыңыз, содан кейін сол оқиға–пайдаланушы нәтижесі үшін Bucket Shuffle мен тұрақты Colocate Join салыстырдыңыз. Жоғарыдағы тоқтату және қайта бастау ұяшықтары оқшауланған 9-модуль кластерін кейін тексеруге сақтайды.

Ресми анықтамалар: [FE жүктемесін теңгеру](https://doris.apache.org/docs/4.x/admin-manual/cluster-management/load-balancing/) · [Икемді масштабтау](https://doris.apache.org/docs/4.x/admin-manual/cluster-management/elastic-expansion/) · [Реплика басқаруы](https://doris.apache.org/docs/4.x/admin-manual/maint-monitor/tablet-repair-and-balance/) · [Join стратегиялары](https://doris.apache.org/docs/4.x/query-data/join/) · [Colocation Join](https://doris.apache.org/docs/4.x/query-acceleration/colocation-join/)

<a id="unit-u.7f1a09c0725547738766ddb5bf39c481"></a>

<a id="next"></a>

## Келесі қадам

[9-тест: Apache Doris жүйесінде шардтау, репликация және үлестірілген орындау](quiz09_distribution_and_availability.ipynb) ашыңыз, содан кейін [10-модуль: Apache Doris жүйесінде сақтау шығындары мен деректердің өмірлік циклін басқару](../module10-managing-data/course.md) бөліміне өтіңіз.

10-зертханалық жұмыс бір түйінді сынақ ортасын қайта пайдаланады және 9-модуль кластері жұмыс істеп тұрса, алғашқы ұяшықта тоқтайды. Run All кластерді қайта бастаумен аяқталады, сондықтан 10-зертханалық жұмысты ашпас бұрын жоғарыдағы **Stop the Module 9 cluster** орындаңыз.